# 2. Data Cleaning, Preprocessing & Modeling



In [ ]:
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE

# Load dataset
df = pd.read_csv('../data/customer_churn_data.csv')
df.drop('customerID', axis=1, inplace=True)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(0, inplace=True)



## Feature Engineering
• AvgMonthlySpend = TotalCharges / tenure
• Tenure buckets (New / Medium / Loyal)
• Service count per customer
• Contract duration flags
• High-value customer indicator



In [ ]:
# Feature 1: AvgMonthlySpend
df['AvgMonthlySpend'] = np.where(df['tenure'] > 0, df['TotalCharges'] / df['tenure'], 0)

# Feature 2: Tenure buckets
df['Tenure_Bucket'] = pd.cut(df['tenure'], bins=[-1, 12, 48, 100], labels=['New', 'Medium', 'Loyal'])

# Feature 3: Service count
services = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['Service_Count'] = sum(df[svc].apply(lambda x: 1 if x in ['Yes', 'Yes (Fiber optic/DSL/etc)'] else 0) for svc in services)

# Feature 4: Contract duration flags
df['Is_LongTerm_Contract'] = df['Contract'].apply(lambda x: 1 if x in ['One year', 'Two year'] else 0)

# Feature 5: High-value customer indicator
high_val_thresh = df['MonthlyCharges'].quantile(0.8)
df['High_Value_Customer'] = df['MonthlyCharges'].apply(lambda x: 1 if x >= high_val_thresh else 0)



## Encoding & Scaling



In [ ]:
target = 'Churn'
y = df[target].apply(lambda x: 1 if x=='Yes' else 0)
X = df.drop(target, axis=1)

# Categorical columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

# Encode categorical variables using Label Encoding 
# In a real pipeline, OneHotEncoding might be better, but LabelEncoding is acceptable
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    # Handle mixed types or unseen values by converting to string
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

# Scale numerical columns
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'Service_Count']
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# Save preprocessors
os.makedirs('../models', exist_ok=True)
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('../models/label_encoders.pkl', 'wb') as f:
    pickle.dump(le_dict, f)

# Train Validation Test split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))



## Class Imbalance Handling



In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"Before SMOTE: %s" % sum(y_train))
print(f"After SMOTE:  %s" % sum(y_train_res))



## Modeling



In [ ]:
def evaluate_model(model, X_val, y_val, name="Model"):
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]
    
    print(f"=== {name} ===")
    print("ROC-AUC:", roc_auc_score(y_val, y_prob))
    print("Precision:", precision_score(y_val, y_pred))
    print("Recall:", recall_score(y_val, y_pred))
    print("F1-score:", f1_score(y_val, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_val, y_pred))
    print("\n")
    return y_prob

# Logistic Regression
lr = LogisticRegression()
lr.fit(X_train_res, y_train_res)
evaluate_model(lr, X_val, y_val, "Logistic Regression")

# Random Forest
rf = RandomForestClassifier(random_state=42, max_depth=6)
rf.fit(X_train_res, y_train_res)
evaluate_model(rf, X_val, y_val, "Random Forest")

# XGBoost
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_res, y_train_res)
xgb_val_prob = evaluate_model(xgb_model, X_val, y_val, "XGBoost")



## Threshold Optimization



In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_val, xgb_val_prob)
f1_scores = 2*precisions*recalls / (precisions+recalls+1e-10)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"Optimal Threshold for F1-score: {optimal_threshold:.4f}")

# Final prediction with optimal threshold
y_val_opt_pred = (xgb_val_prob >= optimal_threshold).astype(int)
print("With optimal threshold:")
print("ROC-AUC:", roc_auc_score(y_val, xgb_val_prob))
print("Precision:", precision_score(y_val, y_val_opt_pred))
print("Recall:", recall_score(y_val, y_val_opt_pred))
print("F1-score:", f1_score(y_val, y_val_opt_pred))

# Evaluate on Test set
xgb_test_prob = xgb_model.predict_proba(X_test)[:, 1]
y_test_pred = (xgb_test_prob >= optimal_threshold).astype(int)
print("\n--- Test Set Evaluation (XGBoost) ---")
print("ROC-AUC:", roc_auc_score(y_test, xgb_test_prob))
print("F1-score:", f1_score(y_test, y_test_pred))

# Save best model logic
with open('../models/xgboost_best.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)
    
# Store final columns required
with open('../models/columns.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)

